###Bronze Ingestion: Automated (Databricks Job version)

Runs once per execution, the Databricks Job schedule handles repetition,
not an internal loop (unlike the original local script). Reads the
VesselAPI key from Databricks Secrets instead of a local .env file, and
writes directly to the Unity Catalog Volume instead of local disk.

In [0]:
import requests
import json
import time
from datetime import datetime, timezone

# API key read securely from Databricks Secrets. not hardcoded or
# stored in notebook code or git history
API_KEY = dbutils.secrets.get(scope="vessel-pipeline-secrets", key="vesselapi-key")

# Volume path replaces the old local BRONZE_DIR. This is the
# permanent, cloud backed storage location
BRONZE_VOLUME_PATH = "/Volumes/logistics_pipeline/bronze/vessel_positions_raw"

POLL_INTERVAL_SECONDS = 300  # Job's scheduled run frequency

# Port of Rotterdam bounding box
BBOX = {
    "filter.latBottom": 51.85,
    "filter.latTop": 52.05,
    "filter.lonLeft": 3.95,
    "filter.lonRight": 4.20,
}

BASE_URL = "https://api.vesselapi.com/v1/location/vessels/bounding-box"
MAX_RETRIES = 3
MAX_PAGES = 15
PAGE_SIZE = 50

## Time window and single page fetch logic

builds a time.from/time.to window
matching the poll interval and fetches a single page with Retry After aware
backoff on rate limits.

In [0]:
def _build_time_window() -> dict:
    # without an explicit window, the endpoint defaults to a 2-hour lookback,
    # causing massive duplicate volume
    #Needs only data matching the poll interval.
    now = datetime.now(timezone.utc)
    time_from = now.timestamp() - POLL_INTERVAL_SECONDS
    return {
        "time.from": datetime.fromtimestamp(time_from, tz=timezone.utc)
            .strftime("%Y-%m-%dT%H:%M:%S.000Z"),
        "time.to": now.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
    }


def _get_page(params: dict) -> dict | None:
    # Retry After aware backoff on 429s, per VesselAPI docs avoids
    # tripping the sustained abuse block that suspends the key for 6 hours
    headers = {"Authorization": f"Bearer {API_KEY}"}
    wait_seconds = 1

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(BASE_URL, headers=headers, params=params, timeout=30)

            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", wait_seconds))
                print(f"[RATE LIMITED] attempt {attempt}/{MAX_RETRIES}, waiting {retry_after}s")
                time.sleep(retry_after)
                wait_seconds *= 2
                continue

            response.raise_for_status()
            return response.json()

        except requests.exceptions.RequestException as e:
            print(f"[ERROR] API request failed (attempt {attempt}/{MAX_RETRIES}): {e}")
            time.sleep(wait_seconds)
            wait_seconds *= 2

    print("[ERROR] All retry attempts exhausted for this page.")
    return None

## Fetch all pages, then write to the Bronze Volume

unlike during testing, Instead of writing local JSON files, this writes directly to
the Unity Catalog Volume using Python's standard file I/O , the Volume
path behaves like a regular filesystem path from the driver's perspective.

In [0]:
def fetch_vessels() -> dict:
    # Same pagination logic validated during local testing, follows
    # nextToken until exhausted, reusing the same time window across pages
    all_vessels: list[dict] = []
    time_window = _build_time_window()
    params = dict(BBOX)
    params.update(time_window)
    params["pagination.limit"] = PAGE_SIZE
    page_count = 0

    while page_count < MAX_PAGES:
        page_count += 1
        page = _get_page(params)

        if page is None:
            print(f"[ERROR] Page {page_count} failed - returning what we have so far.")
            break

        vessels = page.get("vessels", [])
        all_vessels.extend(vessels)

        next_token = page.get("nextToken")
        if not next_token:
            break

        params = dict(BBOX)
        params.update(time_window)
        params["pagination.limit"] = PAGE_SIZE
        params["pagination.nextToken"] = next_token

    if page_count >= MAX_PAGES:
        print(f"[WARNING] Hit MAX_PAGES cap ({MAX_PAGES}) - results may be INCOMPLETE.")

    print(f"[PAGINATION] collected {len(all_vessels)} vessels across {page_count} page(s)")
    return {"vessels": all_vessels}


def write_batch_to_volume(data: dict) -> None:
    # Volume paths behave like a normal filesystem from the driver node,
    # so standard Python file I/O works directly. Spark is not needed for
    # this simple write
    if not data or not data.get("vessels"):
        print("[SKIP] No vessel data to write this cycle.")
        return

    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    filepath = f"{BRONZE_VOLUME_PATH}/{ts}.json"

    with open(filepath, "w") as f:
        json.dump(data, f)

    vessel_count = len(data.get("vessels", []))
    print(f"[BRONZE WRITE] {vessel_count} vessels -> {filepath}")


# Run once, the Databricks Job schedule handles repetition, not an
# internal loop like the original local script
vessel_data = fetch_vessels()
write_batch_to_volume(vessel_data)